In [1]:
# 1. Wipe the directory if running interactively just to be safe
!rm -rf /kaggle/working/Structure-Aware-Graph-Transformer

# 2. Clone repo
!git clone https://github.com/BorgwardtLab/SAT.git /kaggle/working/Structure-Aware-Graph-Transformer

# 3. PyTorch 2.x Compatibility Patch
file_path_models = '/kaggle/working/Structure-Aware-Graph-Transformer/sat/models.py'
with open(file_path_models, 'r') as file:
    code_models = file.read()
if "batch_first" not in code_models:
    code_models = code_models.replace("self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)", 
                                      "encoder_layer.self_attn.batch_first = False\n        self.encoder = GraphTransformerEncoder(encoder_layer, num_layers)")
    with open(file_path_models, 'w') as file:
        file.write(code_models)

# 4. NOVEL FEATURE INJECTION: Adaptive Spectral Selection
patch_code_class = '''
# --- EXTENSION 4: ADAPTIVE POSITIONAL ENCODING ---
class AdaptivePEProjector(nn.Module):
    """
    Dynamically weights positional encoding frequencies (like Random Walk steps) 
    based on their learned relevance to the specific graph structure.
    """
    def __init__(self, pe_dim, embed_dim):
        super().__init__()
        # A gating mechanism to learn the importance of each PE dimension
        self.frequency_gate = nn.Sequential(
            nn.Linear(pe_dim, pe_dim),
            nn.Sigmoid()
        )
        # The final projection to match the Transformer's expected dimension
        self.projector = nn.Linear(pe_dim, embed_dim)

    def forward(self, pe):
        # 1. Calculate importance weights (0 to 1) for each PE dimension
        gate_weights = self.frequency_gate(pe)
        # 2. Modulate the original PE: enhance important features, mute noise
        adaptive_pe = pe * gate_weights
        # 3. Project to the required embedding dimension
        return self.projector(adaptive_pe)
# -------------------------------------------------
'''

with open(file_path_models, 'r') as file:
    code_models = file.read()

if "AdaptivePEProjector" not in code_models:
    # Insert class definition at the top, just under imports
    # FIX 1: Match "from torch import nn" exactly as it is in SAT's original models.py
    code_models = code_models.replace("from torch import nn", "from torch import nn\n" + patch_code_class)
    
    # Replace old static projector with adaptive projector
    # FIX 2: Fixed the whitespace padding so Python won't throw an IndentationError!
    code_models = code_models.replace(
        "self.embedding_abs_pe = nn.Linear(abs_pe_dim, d_model)",
        "# --- ACTIVATING ADAPTIVE SPECTRAL SELECTION ---\n" \
        "            self.embedding_abs_pe = AdaptivePEProjector(abs_pe_dim, d_model)\n" \
        "            # ----------------------------------------------"
    )

    with open(file_path_models, 'w') as file:
        file.write(code_models)

print("Repository cloned and Adaptive Spectral Selection Architecture injected successfully! ✅")


Cloning into '/kaggle/working/Structure-Aware-Graph-Transformer'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (64/64), done.
remote: Total 91 (delta 38), reused 71 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 2.63 MiB | 15.31 MiB/s, done.
Resolving deltas: 100% (38/38), done.
Repository cloned and Adaptive Spectral Selection Architecture injected successfully! ✅


In [2]:
!pip install torch_geometric ogb performer-pytorch einops -q
!pip install torch-scatter torch-sparse -f https://data.pyg.org/whl/torch-2.10.0+cu128.html -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 24.2 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 78.8/78.8 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 64.0 MB/s eta 0:00:0000:010:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 101.3 MB/s eta 0:00:0000:01


In [3]:
!cd /kaggle/working/Structure-Aware-Graph-Transformer && PYTHONPATH=/kaggle/working/Structure-Aware-Graph-Transformer python experiments/train_zinc.py \
    --seed 0 \
    --gnn-type mpnn \
    --use-edge-attr \
    --k-hop 3 \
    --se gnn \
    --abs-pe rw \
    --abs-pe-dim 20 \
    --epochs 2000 \
    --batch-size 128 \
    --lr 0.001 \
    --weight-decay 1e-5 \
    --dropout 0.2 \
    --num-layers 6 \
    --num-heads 8 \
    --dim-hidden 64 \
    --outdir /kaggle/working/results_ultimate_adaptive


Namespace(seed=0, dataset='ZINC', num_heads=8, num_layers=6, dim_hidden=64, dropout=0.2, epochs=2000, lr=0.001, weight_decay=1e-05, batch_size=128, abs_pe='rw', abs_pe_dim=20, outdir='/kaggle/working/results_ultimate_adaptive/ZINC/seed0/edge_attr/rw_20/mpnn_3_0.2_0.001_1e-05_6_8_64_BN', warmup=5000, layer_norm=False, use_edge_attr=True, edge_dim=32, gnn_type='mpnn', k_hop=3, global_pool='mean', se='gnn', use_cuda=True, batch_norm=True, save_logs=True)
Extracting ../datasets/ZINC/molecules.zip
Processing...
Processing test dataset: 100%|████████████| 1000/1000 [00:00<00:00, 7961.16it/s]
Done!
/kaggle/working/Structure-Aware-Graph-Transformer/experiments/train_zinc.py:207: UserWarning: 'data.DataLoader' is deprecated, use 'loader.DataLoader' instead
  train_loader = DataLoader(train_dset, batch_size=args.batch_size,
Data(x=[29], edge_index=[2, 64], edge_attr=[64], y=[1, 1], complete_edge_index=[2, 841], degree=[29])
/kaggle/working/Structure-Aware-Graph-Transformer/experiments/train_zinc